# Emergent Vocabulary Analysis

Loads a trained sender/receiver pair and analyzes:
1. **Topographic similarity** — does the language reflect object structure?
2. **Entropy-adaptive segmentation** — what 'words' emerge spontaneously?
3. **Vocabulary statistics** — frequency distribution, Zipf fit
4. **Attribute alignment** — do specific words encode specific attributes?
5. **Qualitative examples** — what does the sender say about each object?

In [ ]:
import sys
sys.path.insert(0, '../..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

from emergent_comm.config import ExperimentConfig
from emergent_comm.environment.entities import all_objects, COLOR_NAMES, SHAPE_NAMES, SIZE_NAMES
from emergent_comm.agents.sender import Sender
from emergent_comm.agents.receiver import Receiver
from emergent_comm.evaluation.compositionality import evaluate_compositionality
from emergent_comm.evaluation.vocabulary import collect_vocabulary, print_vocabulary_report

## 1. Load Model

In [ ]:
cfg = ExperimentConfig()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

sender = Sender(
    n_colors=cfg.env.n_colors, n_shapes=cfg.env.n_shapes, n_sizes=cfg.env.n_sizes,
    bb_cfg=cfg.backbone, dec_cfg=cfg.decoder, agent_cfg=cfg.agent,
).to(device)

receiver = Receiver(
    n_colors=cfg.env.n_colors, n_shapes=cfg.env.n_shapes, n_sizes=cfg.env.n_sizes,
    enc_cfg=cfg.encoder, bb_cfg=cfg.backbone,
).to(device)

ckpt = torch.load('../../checkpoints/best.pt', map_location=device)
sender.load_state_dict(ckpt['sender'])
receiver.load_state_dict(ckpt['receiver'])
sender.eval(); receiver.eval()
print(f'Loaded checkpoint from step {ckpt["step"]} (val_acc={ckpt["val_acc"]:.3f})')

## 2. Collect Messages for All Objects

In [ ]:
object_pool = all_objects(cfg.env.n_colors, cfg.env.n_shapes, cfg.env.n_sizes)

objects_tuples = []
messages_bytes = []

with torch.no_grad():
    for obj in object_pool:
        attrs = torch.tensor([obj.to_vector()], dtype=torch.long, device=device)
        out = sender(attrs)
        msg = out['message_bytes'][0].cpu().tolist()
        # Strip trailing EOS
        eos = cfg.agent.eos_byte
        if eos in msg:
            msg = msg[:msg.index(eos)]
        objects_tuples.append(obj.to_vector())
        messages_bytes.append(msg)

print(f'Collected {len(messages_bytes)} messages')
print(f'Mean message length: {np.mean([len(m) for m in messages_bytes]):.1f} bytes')
print(f'Unique messages: {len(set(tuple(m) for m in messages_bytes))}')

## 3. Topographic Similarity

In [ ]:
comp_results = evaluate_compositionality(
    objects_tuples, messages_bytes,
    n_topsim_samples=cfg.eval.n_topsim_samples,
    seed=cfg.env.seed,
)

print(f'Topographic Similarity (topsim): {comp_results["topsim"]:.4f}')
print('  > 0.3 = weak compositionality')
print('  > 0.6 = strong compositionality')
print()
for k, v in comp_results.items():
    if 'max_disentanglement' in k:
        attr_idx = int(k.split('attr')[1].split('_')[0])
        attr_name = ['color', 'shape', 'size'][attr_idx]
        print(f'  {attr_name} max positional disentanglement: {v:.4f}')

## 4. Vocabulary Analysis (Entropy-Adaptive Segmentation)

In [ ]:
vocab_results = collect_vocabulary(
    objects_tuples, messages_bytes, seg_cfg=cfg.segmentation
)
print_vocabulary_report(vocab_results)

## 5. Frequency Distribution (Zipf Plot)

In [ ]:
freqs = sorted(vocab_results['frequency_distribution'].values(), reverse=True)
ranks = list(range(1, len(freqs) + 1))

plt.figure(figsize=(7, 4))
plt.loglog(ranks, freqs, 'o-', markersize=4)
plt.xlabel('Rank')
plt.ylabel('Frequency')
plt.title(f'Word Frequency Distribution (Zipf corr={vocab_results["zipf_correlation"]:.3f})')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('zipf_plot.png', dpi=150)
plt.show()

## 6. Qualitative Examples

In [ ]:
print(f'{"Object":<35} {"Message (bytes)"}' )
print('-' * 70)
for obj, msg in zip(object_pool[:20], messages_bytes[:20]):
    printable = ''.join(chr(b) if 32 <= b < 127 else f'[{b}]' for b in msg)
    print(f'{obj.to_label():<35} {printable}')

## 7. Heatmap: Which bytes encode which attributes?

In [ ]:
disent = comp_results['positional_disentanglement']
max_pos = 8
n_attrs = 3
attr_names = ['color', 'shape', 'size']

matrix = np.zeros((max_pos, n_attrs))
for pos in range(max_pos):
    for attr in range(n_attrs):
        key = f'pos{pos}_attr{attr}'
        matrix[pos, attr] = disent.get(key, 0.0)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(matrix, cmap='Blues', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(n_attrs))
ax.set_xticklabels(attr_names)
ax.set_yticks(range(max_pos))
ax.set_yticklabels([f'pos {i}' for i in range(max_pos)])
ax.set_title('Positional Disentanglement\n(how much each byte position encodes each attribute)')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig('disentanglement_heatmap.png', dpi=150)
plt.show()